### instalar librerias

esto solo hay que correrlo una vez

In [39]:

!pip uninstall -y sentence-transformers datasets pyarrow tensorflow tensorflow-intel keras
!pip install -q "sentence-transformers==2.7.0" transformers torch faiss-cpu numpy pandas
!pip install -q "transformers>=4.40,<5.0" "tokenizers>=0.19,<0.21" sentencepiece protobuf


Found existing installation: sentence-transformers 2.7.0
Uninstalling sentence-transformers-2.7.0:
  Successfully uninstalled sentence-transformers-2.7.0


### imports

In [40]:
import os
import re
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore') 

print("listo")


listo


In [41]:
RUTA_CORPUS = "data/corpus_resenas.csv"

df = pd.read_csv(RUTA_CORPUS)
print(f"{len(df)} reseñas cargadas")
df[["texto", "tipo_lugar", "calificacion", "lugar"]].head(3)


5018 reseñas cargadas


,texto,tipo_lugar,calificacion,lugar
0,"Tour fabuloso, bellísimo el bosque nuboso de M...",parque,5.0,Excursión a los puentes colgantes de Monteverde
1,Vale la pena ir a ver los puentes colgantes de...,parque,4.0,Excursión a los puentes colgantes de Monteverde
2,"La actividad es muy bonita, sin embargo hay do...",parque,4.0,Excursión a los puentes colgantes de Monteverde


### chunking

un chunk es cada pedacito en el que partimos el texto antes de convertirlo en vector. si
metemos la reseña completa sin partir se pierde precision en la busqueda cuando el texto es
largo, pero tampoco conviene pasarse partiendo porque se pierde contexto.

como nuestras reseñas ya son cortas la diferencia no es tan grande como con un texto largo,
pero se nota en la cantidad de chunks que salen de cada una

In [ ]:
def chunking_oraciones(texto, oraciones_por_chunk=3, overlap_oraciones=1):
    # separa por punto, signo de exclamacion o interrogacion
    oraciones = re.split(r'(?<=[.!?])\s+', texto.strip())
    oraciones = [o.strip() for o in oraciones if o.strip()]

    chunks = []
    i = 0
    while i < len(oraciones):
        grupo = oraciones[i:i + oraciones_por_chunk]
        chunk = " ".join(grupo)
        if chunk:
            chunks.append(chunk)
        i += oraciones_por_chunk - overlap_oraciones  
    return chunks


def chunking_parrafos(texto, min_longitud=50):
    # separa por parrafos (doble salto de linea)
    parrafos = re.split(r'\n\s*\n', texto)
    parrafos = [p.strip() for p in parrafos if p.strip()]

    # si un parrafo queda muy corto lo pega con el siguiente para que no quede un chunk vacio de info
    chunks = []
    buffer = ""
    for p in parrafos:
        if len(buffer) + len(p) < min_longitud * 3:
            buffer += " " + p
        else:
            if buffer.strip():
                chunks.append(buffer.strip())
            buffer = p
    if buffer.strip():
        chunks.append(buffer.strip())

    return chunks


In [ ]:
# recorre el corpus reseña por reseña y le va aplicando el chunking
def aplicar_chunking(df, funcion, **kwargs):
    resultado = []
    for idx, fila in df.iterrows():
        for fragmento in funcion(str(fila["texto"]), **kwargs):
            resultado.append({
                "texto": fragmento,
                "lugar": fila["lugar"],
                "tipo_lugar": fila["tipo_lugar"],
                "calificacion": fila["calificacion"],
                "polaridad": fila["polaridad"],
                "fuente": fila["fuente"],
            })
    return resultado


In [44]:
# comparamos las dos estrategias sobre todo el corpus
chunks_oraciones = aplicar_chunking(df, chunking_oraciones, oraciones_por_chunk=3, overlap_oraciones=1)
chunks_parrafos = aplicar_chunking(df, chunking_parrafos)

for nombre, lista in [("oraciones", chunks_oraciones), ("parrafos", chunks_parrafos)]:
    largos = [len(c["texto"]) for c in lista]
    print(f"{nombre}: {len(lista)} chunks, promedio {np.mean(largos):.0f} caracteres")


oraciones: 9251 chunks, promedio 182 caracteres
parrafos: 5018 chunks, promedio 270 caracteres


In [45]:
CHUNKS = chunks_parrafos
print(f"chunks finales: {len(CHUNKS)}")
print(CHUNKS[0])


chunks finales: 5018
{'texto': 'Tour fabuloso, bellísimo el bosque nuboso de Monteverde con el circuito de los puentes colgantes. Tras el almuerzo visitamos el ranario de la zona, donde pudimos observar gran variedad de sapos y ranitas con la ayuda del guía del lugar. Nuestro guía y conductor, Arturo, muy amable, atento y simpático. Recomendamos 100% esta actividad.', 'lugar': 'Excursión a los puentes colgantes de Monteverde', 'tipo_lugar': 'parque', 'calificacion': 5.0, 'polaridad': 'positiva', 'fuente': 'civitatis'}


### embeddings

un embedding es pasar el texto a un vector de numeros que representa el significado. la idea
es que textos parecidos en significado queden con vectores cerca entre si.


In [ ]:
from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("modelo cargado, dimension:", modelo_embeddings.get_sentence_embedding_dimension())

textos = [c["texto"] for c in CHUNKS]
embeddings = modelo_embeddings.encode(textos, show_progress_bar=True)  
print("embeddings listos:", embeddings.shape)


modelo cargado, dimension: 384


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

embeddings listos: (5018, 384)


### indice FAISS

FAISS es para no tener que comparar la pregunta contra todos los embeddings a mano, uno por
uno. usamos IndexFlatIP con los vectores normalizados, que da lo mismo que buscar por
similitud coseno 

In [ ]:
import faiss

dimension = embeddings.shape[1]
indice = faiss.IndexFlatIP(dimension)

faiss.normalize_L2(embeddings)  # normalizar antes de usar IndexFlatIP
indice.add(embeddings)

print(f"indice con {indice.ntotal} vectores")


indice con 5018 vectores


### buscar chunks relevantes

esta es la parte de "retrieval": convierte la pregunta en embedding igual que a los chunks, y
le pide a FAISS los top_k mas parecidos

In [48]:
def buscar_chunks_relevantes(pregunta, top_k=3):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    faiss.normalize_L2(embedding_pregunta)

    scores, indices = indice.search(embedding_pregunta, top_k)

    resultados = []
    for score, idx in zip(scores[0], indices[0]):
        chunk = CHUNKS[idx]
        resultados.append({
            "chunk": chunk["texto"],
            "lugar": chunk["lugar"],
            "tipo_lugar": chunk["tipo_lugar"],
            "calificacion": chunk["calificacion"],
            "score": float(score),
        })
    return resultados


# prueba rapida a ver si trae algo coherente
for r in buscar_chunks_relevantes("hoteles con buena atencion al cliente"):
    print(f"[{r['score']:.3f}] ({r['tipo_lugar']} - {r['lugar']}) {r['chunk'][:120]}...")


[0.854] (hotel - Hotel Alajuela City) Excelente Hotel muy limpio y el personal muy amable lo recomiendo...
[0.847] (hotel - Hotel Park View) Muy buen hotel agradable...
[0.845] (hotel - Hotel Alajuela City) Hotel boutique muy bien acondicionado...


### generar la respuesta

esta es la parte de "generation": le pasamos al LLM el contexto que encontramos mas  terminamos usando
Qwen2.5-1.5B-Instruct, que es multilingue de verdad y sigue instrucciones tipo chat mejor.


In [49]:
from transformers import pipeline

# descarga el modelo
generador_local = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_tokens=256
)
print("generador listo")


generador listo


In [50]:
def generar_respuesta(contexto, pregunta):
    mensajes = [
        {"role": "system", "content": "Respondes preguntas sobre turismo en Costa Rica basandote SOLO en el contexto que se te da. Da respuestas completas y explicadas, no solo una linea. Si no encontras la respuesta en el contexto, decis que no tenes informacion suficiente."},
        {"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"}
    ]
    # max_new_tokens mas alto para que no corte la respuesta a la mitad
    salida = generador_local(mensajes, max_new_tokens=400)
    return salida[0]["generated_text"][-1]["content"]


### rag completo



In [51]:
def rag_completo(pregunta, top_k=3):
    print(f"\nPREGUNTA: {pregunta}")

    resultados = buscar_chunks_relevantes(pregunta, top_k)
    for r in resultados:
        print(f"  - ({r['tipo_lugar']} - {r['lugar']}, score {r['score']:.3f}) {r['chunk'][:80]}...")

    contexto = "\n\n".join(
        f"({r['tipo_lugar']} - {r['lugar']}, {r['calificacion']} estrellas): {r['chunk']}"
        for r in resultados
    )

    respuesta = generar_respuesta(contexto, pregunta)
    print(f"\nRESPUESTA: {respuesta}")
    return respuesta


### pruebas

le hacemos algunas preguntas para ver como responde

In [52]:
preguntas = [
    "Que hoteles tienen buena atencion al cliente?",
    "Recomiendame un parque nacional con senderos bonitos",
    "Que opinan de los museos en San Jose?",
    "Que diferencia una reseña de un parque de una de un hotel?",
    "De que se quejan en los mercados artesanales?",
]

for p in preguntas:
    rag_completo(p)



PREGUNTA: Que hoteles tienen buena atencion al cliente?
  - (hotel - Hotel Alajuela City, score 0.825) Excelente Hotel muy limpio y el personal muy amable lo recomiendo...
  - (hotel - Hotel Courtyard de Marriott • Alajuela, score 0.806) Perfecto lugar para estar Excelente Hotel...
  - (hotel - Hotel Alajuela City, score 0.802) Hotel boutique muy bien acondicionado...

RESPUESTA: Según el contexto proporcionado, los hoteles con buena atención al cliente son:

1. **Hotel Alajuela City (5.0 estrellas)**: El hotel está calificado como excelente y es mencionado como un lugar donde el personal es muy amable.

2. **Hotel Courtyard de Marriott • Alajuela (5.0 estrellas)**: Este también tiene una alta nota de 5.0 y se describe como un buen lugar para estar.

Ambos hoteles reciben calificaciones altas por su atención al cliente, lo cual sugiere que tienen buenas relaciones entre el personal del establecimiento y sus huéspedes. Sin embargo, hay un pequeño detalle importante: aunque ambos hotele